# delhi-weather: forecasting (online, custom)

**Use case:** `usecase-type/online/timeseries-forecasting/notebook` — request/response scoring through an endpoint with the custom notebook (local shared engine).

**Training:** Local custom training and held-out evaluation with the shared engine; the same scenario renders as a quality-gated Azure ML v2 pipeline before registration.

**Serving:** Local `online-test` contract check; Azure ML managed online endpoint via `serving-render`, `serving-deploy` and `serving-invoke`.

**Status:** Executable notebook templates: local steps run offline after data approval; cloud steps are explicit, charged operations behind switches that default to false.

Kaggle-only source: `sumanthvrao/daily-climate-time-series-data`.
Dataset status: **configured**.
Daily mean temperature; weather covariates are omitted because their future values are not known at forecast origin. Last two 14-day windows of the labeled training file are held out. Verify upstream Weather Underground attribution for intended use.

**Prerequisites**

- Review the scenario's dataset status, license and notes; gated datasets stay blocked until approved.
- Install the factory extras `train` (plus `vision` for images, `azure` for Azure ML steps and `streaming` for Event Hubs consumers).
- For Azure steps: `user-config/runtime.local.json` naming an existing workspace, compute, identity and storage selection.
- Endpoint quota; deployed endpoints bill while running, so delete or scale them to zero when done.

**Limitations**

- The custom seasonal-naive baseline only forecasts timestamps inside its fitted horizon window.
- Templates are not evidence of a completed Kaggle, Azure or Databricks run.

This notebook is an executable template, not evidence of a completed Kaggle or Azure run.
Install the factory's declared extras into this kernel separately. No packages,
credentials, cloud resources, or competition terms are installed/accepted by this notebook.
All execution switches default to false. Run **either** SDK v2 **or** CLI v2 submission, not both.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "accelerator" / "src" / "ml_model_factory" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the 50-ml-model-factory checkout")
SOURCE = ROOT / "accelerator" / "src"
sys.path.insert(0, str(SOURCE))
from ml_model_factory.config import load_json, validate_scenario, validate_runtime, write_json

SCENARIO = ROOT / "user-config" / "model" / "scenarios" / "delhi-weather.json"
scenario = validate_scenario(load_json(SCENARIO))
MODE = "custom"
PATTERN = "online"
TECHNOLOGY = "notebook"
RUNTIME = Path("user-config/runtime.local.json")
if not RUNTIME.is_absolute():
    RUNTIME = ROOT / RUNTIME
WORK = ROOT / "ml-environment" / "outputs" / scenario["name"] / "online-custom"
RAW = ROOT / "data" / "in" / scenario["name"]
PREPARED = ROOT / "data" / "out" / scenario["name"] / "online-custom" / "prepared"
MODEL = WORK / "model"
EVALUATION = WORK / "evaluation"
RENDERED = WORK / "azureml"
INFERENCE = WORK / PATTERN
REQUESTS = INFERENCE / "requests"
MODEL_NAME = scenario.get("model_name", scenario["name"])
MODEL_ID = ""  # immutable azureml:<name>:<version> after quality-gated registration
RUN_INGEST = False
RUN_PREPARE = False
RUN_LOCAL_TRAIN = False
SAMPLE_REQUESTS = False
RUN_ONLINE_TEST = False
USE_LAKE = False
RUN_LAKE_TRAIN = False
RENDER_AZURE = False
SUBMIT_SDK = False
SUBMIT_CLI = False
REGISTER_MODEL = False
RENDER_ONLINE = False
DEPLOY_ONLINE = False
INVOKE_ONLINE = False
DELETE_ONLINE = False
PIPELINE_JOB_NAME = ""
ONLINE_ENDPOINT = None
LAKE_CONFIG = ROOT / "user-config" / "lake.local.json"
LAKE_ROOT = ROOT / "data" / "out" / "lake-data"
EFFECTIVE_RUNTIME = RUNTIME
FACTORY_ENV = os.environ.copy()
FACTORY_ENV["PYTHONPATH"] = str(SOURCE) + os.pathsep + FACTORY_ENV.get("PYTHONPATH", "")
FACTORY_ENV.setdefault("MLFLOW_TRACKING_URI", (ROOT / "ml-environment" / "mlruns").as_uri())

def factory(*arguments):
    command = [sys.executable, "-m", "ml_model_factory", *map(str, arguments)]
    subprocess.run(command, cwd=ROOT, env=FACTORY_ENV, check=True)


## Kaggle ingestion and reproducible preparation

Review the source license and any competition rules first. Required-selection and
license-review gates block ingestion. Do not clear them without documented approval.
The download manifest records file hashes and pinned dataset version. Existing nonempty
outputs are never silently replaced. Titanic preparation additionally writes `titanic.parquet`.
Forecasting uses chronological holdouts, not random rows.


In [ ]:
if RUN_INGEST:
    factory("ingest", "--scenario", SCENARIO, "--output", RAW)
INPUT = RAW / scenario["dataset"]["file"]
if RUN_PREPARE:
    factory("prepare", "--scenario", SCENARIO, "--input", INPUT, "--output", PREPARED)
if (PREPARED / "manifest.json").exists():
    print(json.dumps(load_json(PREPARED / "manifest.json"), indent=2))


## Custom training, held-out evaluation, and Responsible AI

The custom route fits only the training split and exports an MLflow pyfunc model.
Tabular classification/regression provide held-out errors, cohorts and permutation
importance; forecasting preserves time order. Vision produces task-specific metrics
and image/class diagnostics, not an unsupported Azure tabular RAI dashboard.
Small CPU vision limits demonstrate optimization, not production model quality.


In [ ]:
if RUN_LOCAL_TRAIN:
    factory("train", "--scenario", SCENARIO, "--prepared", PREPARED, "--model-output", MODEL)
    factory("evaluate", "--scenario", SCENARIO, "--prepared", PREPARED, "--model", MODEL, "--output", EVALUATION)
if (EVALUATION / "responsible-ai.json").exists():
    print(json.dumps(load_json(EVALUATION / "responsible-ai.json"), indent=2))


## Unlabeled requests from the held-out split

`sample-requests` writes demonstration requests without labels; `online-request.json` is the endpoint payload.
`labels.parquet` keeps held-out truth separately for monitoring demos, and forecasting
also writes `history.parquet` for AutoML scoring. Never send labels to a model.


In [ ]:
if SAMPLE_REQUESTS:
    factory("sample-requests", "--scenario", SCENARIO, "--prepared", PREPARED, "--output", REQUESTS, "--rows", 10)
if (REQUESTS / "manifest.json").exists():
    print(json.dumps(load_json(REQUESTS / "manifest.json"), indent=2))


## Local online contract test

Runs the same request parser and scoring service as the Azure ML scoring entry point
in-process (no Docker, no network) and checks one prediction per request row.


In [ ]:
if RUN_ONLINE_TEST:
    factory("online-test", "--scenario", SCENARIO, "--model", MODEL,
            "--request", REQUESTS / "online-request.json", "--output", INFERENCE / "online-response.json")
if (INFERENCE / "online-response.json").exists():
    print(json.dumps(load_json(INFERENCE / "online-response.json"), indent=2))


## Versioned project/use-case lake (optional)

Set the project/environment, dataset version, snapshot ID and unique run ID in
`user-config/lake.local.json`. `lake-plan` is read-only. `lake-train` publishes source, gold snapshots,
models and evaluations locally with completion manifests; it does not upload data.
Keep unlabeled inference requests and observed feedback separate. Use `lake-infer` with
an immutable model version and unique request IDs, and `lake-feedback` for reviewed labels.
Streaming checkpoints are independent of execution run IDs and model versions.

Azure preparation uses a per-run working area and records the snapshot reference;
it does not claim that binding an output URI alone publishes an immutable snapshot.


In [ ]:
if USE_LAKE:
    factory("lake-plan", "--scenario", SCENARIO, "--config", LAKE_CONFIG)
    if RENDER_AZURE:
        runtime_with_lake = validate_runtime(load_json(RUNTIME))
        runtime_with_lake["lake"] = load_json(LAKE_CONFIG)
        EFFECTIVE_RUNTIME = WORK / "runtime.local.json"
        write_json(EFFECTIVE_RUNTIME, runtime_with_lake)
if RUN_LAKE_TRAIN:
    if not USE_LAKE or MODE != "custom":
        raise ValueError("Local lake training requires USE_LAKE and custom mode")
    factory("lake-train", "--scenario", SCENARIO, "--config", LAKE_CONFIG,
            "--root", LAKE_ROOT, "--input", INPUT)


## Render portable Azure ML v2 jobs (offline)

`user-config/runtime.local.json` must name existing subscription, workspace and compute resources and a
resolvable input data path. Resource discovery is read-only. Rendering creates files,
not Azure resources. Submission incurs compute charges. Image AutoML additionally
requires a GPU-compatible compute target and the prepared remote image layout above.


In [ ]:
if RENDER_AZURE:
    validate_runtime(load_json(EFFECTIVE_RUNTIME))
    factory("render", "--scenario", SCENARIO, "--runtime", EFFECTIVE_RUNTIME,
            "--mode", MODE, "--output", RENDERED)
JOB_PATH = RENDERED / "pipeline.yml" if (RENDERED / "pipeline.yml").exists() else RENDERED / "job.yml"


## Azure ML SDK v2 submission — alternative A


In [ ]:
if SUBMIT_SDK:
    if SUBMIT_CLI:
        raise ValueError("Choose one submission route to avoid duplicate charged jobs")
    from ml_model_factory.azureml import submit
    runtime = validate_runtime(load_json(EFFECTIVE_RUNTIME))
    if not JOB_PATH.exists():
        raise RuntimeError("Render and review the job before submission")
    # Shared SDK v2 submission binds the configured tenant and rejects unresolved inputs.
    print(submit(JOB_PATH, runtime))


## Azure ML CLI v2 submission — alternative B


In [ ]:
if SUBMIT_CLI:
    if SUBMIT_SDK:
        raise ValueError("Choose one submission route to avoid duplicate charged jobs")
    runtime = validate_runtime(load_json(EFFECTIVE_RUNTIME))
    if not JOB_PATH.exists():
        raise RuntimeError("Render and review the job before submission")
    subprocess.run([sys.executable, str(ROOT / "accelerator" / "scripts" / "azureml_cli.py"),
                    "--runtime", str(EFFECTIVE_RUNTIME), "--job", str(JOB_PATH)],
                   cwd=ROOT, env=FACTORY_ENV, check=True)

# Equivalent factory SDK-backed submission:
# factory("submit", "--runtime", RUNTIME, "--job", JOB_PATH)


## Quality-gated model registration

Only a completed factory pipeline whose `evaluate` step wrote a passing `quality-gate.json`
can register its named model output. Standalone training jobs are not a quality gate.
Set `PIPELINE_JOB_NAME` to the completed job, then copy the printed immutable model ID
into `MODEL_ID`. Registration is not production promotion.


In [ ]:
if REGISTER_MODEL:
    if not PIPELINE_JOB_NAME:
        raise ValueError("Set PIPELINE_JOB_NAME to the completed, evaluated factory pipeline job")
    subprocess.run([sys.executable, str(ROOT / "accelerator" / "scripts" / "azureml_sdk.py"),
                    "--runtime", str(EFFECTIVE_RUNTIME), "register", "--job-name", PIPELINE_JOB_NAME,
                    "--model-name", MODEL_NAME], cwd=ROOT, env=FACTORY_ENV, check=True)
print("MODEL_ID:", MODEL_ID or "not set")


## Azure ML managed online endpoint

`serving-render` chooses no-code MLflow deployment where Azure supports it and a custom
scoring entry point (the same service as the local contract test) otherwise. Deployment
checks the registered model's quality-gate, pipeline, scenario, mode and scope tags and
moves traffic last. `serving-invoke` adapts the factory request to the bundle's documented
contract (AutoML image column, AutoML forecast history). Endpoints bill while deployed:
delete them when finished.


In [ ]:
ONLINE_BUNDLE = INFERENCE / "azureml-online"
if RENDER_ONLINE:
    factory("serving-render", "--scenario", SCENARIO, "--runtime", EFFECTIVE_RUNTIME, "--kind", "online",
            "--model-name", MODEL_NAME, "--mode", MODE, "--output", ONLINE_BUNDLE)
if ONLINE_ENDPOINT is None and (ONLINE_BUNDLE / "manifest.json").exists():
    ONLINE_ENDPOINT = load_json(ONLINE_BUNDLE / "manifest.json")["endpoint_name"]
if DEPLOY_ONLINE:
    if not MODEL_ID:
        raise ValueError("Set MODEL_ID to the immutable registered model version")
    factory("serving-deploy", "--runtime", EFFECTIVE_RUNTIME, "--bundle", ONLINE_BUNDLE,
            "--model-id", MODEL_ID, "--execute")
if INVOKE_ONLINE:
    factory("serving-invoke", "--runtime", EFFECTIVE_RUNTIME, "--kind", "online", "--endpoint", ONLINE_ENDPOINT,
            "--request", REQUESTS / "online-request.json", "--bundle", ONLINE_BUNDLE, "--execute")
if DELETE_ONLINE:
    factory("serving-delete", "--runtime", EFFECTIVE_RUNTIME, "--kind", "online", "--endpoint", ONLINE_ENDPOINT,
            "--execute")


## Monitoring and clean-up

Join predictions with observed labels by `request_id` only after outcomes are known, then use
`python -m ml_model_factory monitor` with explicit reference/current windows (see the
model-factory guide). Predictions are never labels. Delete online endpoints (or Databricks serving endpoints) when the test is over.
